# Multi-modal RAG with LangChain

이 노트북은 PDF 파일에서 텍스트, 이미지, 표를 추출하고 처리하는 multi-modal RAG 시스템을 구현합니다.

## 주요 기능
- PDF에서 텍스트, 표, 이미지 추출
- 이미지와 표를 텍스트로 요약
- Multi-vector retriever를 사용한 효율적인 검색
- GPT-4o를 활용한 multi-modal 답변 생성

# Multi-modal RAG with LangChain & LlamaParse

이 노트북은 LlamaParse를 사용하여 PDF 파일에서 텍스트, 이미지, 표를 추출하고 처리하는 multi-modal RAG 시스템을 구현합니다.

## 주요 기능
- LlamaParse를 사용한 고품질 PDF 파싱
- PDF에서 텍스트, 표, 이미지 추출
- 이미지와 표를 텍스트로 요약
- Multi-vector retriever를 사용한 효율적인 검색
- GPT-4o를 활용한 multi-modal 답변 생성

## 필수 설정
1. **LLAMA_CLOUD_API_KEY**: [LlamaIndex Cloud](https://cloud.llamaindex.ai/)에서 발급
2. **OPENAI_API_KEY**: OpenAI GPT-4o 사용을 위해 필요

.env 파일에 다음과 같이 추가하세요:
```
LLAMA_CLOUD_API_KEY=llx-your-api-key-here
OPENAI_API_KEY=sk-your-api-key-here
```

In [1]:
import os
import base64
from typing import List, Dict, Any
from io import BytesIO
from PIL import Image
import uuid

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain.text_splitter import RecursiveCharacterTextSplitter

# LlamaParse 관련 임포트
from llama_parse import LlamaParse
from llama_index.core import SimpleDirectoryReader
import nest_asyncio

# nest_asyncio 활성화 (Jupyter notebook에서 필요)
nest_asyncio.apply()

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv()

True

## 2. LlamaParse 설정

LlamaParse를 사용하려면 LLAMA_CLOUD_API_KEY가 필요합니다. 
[LlamaIndex Cloud](https://cloud.llamaindex.ai/)에서 API 키를 발급받으세요.

In [2]:
# 형식 비교 함수
def compare_formats():
    """Markdown vs Text 형식 비교"""
    print("=" * 60)
    print("결과 형식 비교")
    print("=" * 60)
    
    print("\n📋 Markdown 형식의 장점:")
    print("✅ 표 구조 명확 식별 (| 구분자)")
    print("✅ 제목 계층 구조 보존 (#, ##, ###)")
    print("✅ 이미지 설명 패턴 일관성")
    print("✅ 구조적 파싱 용이")
    
    print("\n📄 Text 형식의 장점:")
    print("✅ 순수 텍스트로 단순함")
    print("✅ 특수 문자 없음")
    print("✅ 일반 텍스트 처리 파이프라인과 호환")
    
    print("\n💡 권장사항:")
    print("- 표와 이미지가 많은 문서: Markdown")
    print("- 순수 텍스트 문서: Text")
    print("- 구조적 분석 필요: Markdown")

# 실행
compare_formats()

결과 형식 비교

📋 Markdown 형식의 장점:
✅ 표 구조 명확 식별 (| 구분자)
✅ 제목 계층 구조 보존 (#, ##, ###)
✅ 이미지 설명 패턴 일관성
✅ 구조적 파싱 용이

📄 Text 형식의 장점:
✅ 순수 텍스트로 단순함
✅ 특수 문자 없음
✅ 일반 텍스트 처리 파이프라인과 호환

💡 권장사항:
- 표와 이미지가 많은 문서: Markdown
- 순수 텍스트 문서: Text
- 구조적 분석 필요: Markdown


In [3]:
def setup_llama_parse(result_format="markdown"):
    """LlamaParse 설정 및 초기화"""
    # API 키 확인
    llama_api_key = os.getenv('LLAMA_CLOUD_API_KEY')
    if not llama_api_key:
        print("경고: LLAMA_CLOUD_API_KEY가 설정되지 않았습니다.")
        print("LlamaIndex Cloud에서 API 키를 발급받고 .env 파일에 추가해주세요.")
        return None
    
    # 결과 형식에 따른 프롬프트 조정
    if result_format == "markdown":
        system_prompt = """당신은 PDF 문서를 마크다운 형식으로 변환하는 전문가입니다.
다음 지침을 따라 정확하고 구조화된 변환을 수행하세요:
1. 표(table)는 마크다운 표 형식으로 정확히 변환 (| 구분자 사용)
2. 이미지와 차트가 있으면 해당 내용을 상세히 텍스트로 설명
3. 차트의 경우 데이터 트렌드, 주요 수치, 패턴을 포함하여 설명
4. 제목과 구조를 명확히 유지 (#, ##, ### 사용)
5. 텍스트의 의미를 보존하며 정확히 파싱
6. 이미지 설명은 [이미지: 설명내용] 형태로 표시"""
        user_prompt = "이 PDF 문서를 마크다운 형식으로 변환해주세요. 표는 마크다운 테이블로, 이미지는 텍스트 설명으로 변환해주세요."
    else:  # text format
        system_prompt = """당신은 PDF 문서를 구조화된 텍스트로 변환하는 전문가입니다.
다음 지침을 따라 정확한 변환을 수행하세요:

1. 표의 내용을 명확하게 텍스트로 변환 (각 행과 열 구분)
2. 이미지와 차트의 내용을 상세히 텍스트로 설명
3. 제목과 구조를 텍스트로 명확히 표현
4. 모든 정보를 손실 없이 텍스트로 변환"""
        user_prompt = "이 PDF 문서를 구조화된 일반 텍스트로 변환해주세요. 모든 정보를 포함하되 읽기 쉽게 정리해주세요."
    
    # LlamaParse 초기화
    parser = LlamaParse(
        api_key=llama_api_key,
        result_type=result_format,  # "markdown" 또는 "text"
        verbose=True,
        language="ko",
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )
    
    return parser

def extract_pdf_with_llama_parse(pdf_path: str, result_format="markdown") -> List[Document]:
    """LlamaParse를 사용하여 PDF 추출"""
    parser = setup_llama_parse(result_format)
    if not parser:
        return []
    
    try:
        print(f"LlamaParse로 PDF 파싱 중: {pdf_path}")
        print(f"결과 형식: {result_format}")
        print("이미지와 텍스트를 함께 처리하고 있습니다...")
        
        # PDF 파일이 디렉토리가 아닌 단일 파일인 경우 처리
        pdf_dir = os.path.dirname(pdf_path)
        pdf_filename = os.path.basename(pdf_path)
        
        # SimpleDirectoryReader로 파일 읽기
        file_extractor = {".pdf": parser}
        reader = SimpleDirectoryReader(
            input_dir=pdf_dir,
            file_extractor=file_extractor,
            filename_as_id=True,
            required_exts=[".pdf"]
        )
        
        # 특정 파일만 로드
        documents = []
        for file_path in reader.input_files:
            if os.path.basename(file_path) == pdf_filename:
                docs = reader.load_data()
                documents.extend(docs)
                break
        
        print(f"추출 완료: {len(documents)}개 문서")
        
        # LlamaIndex Document를 LangChain Document로 변환
        langchain_docs = []
        for doc in documents:
            langchain_doc = Document(
                page_content=doc.text,
                metadata={
                    "source": doc.metadata.get("file_name", pdf_path),
                    "page": doc.metadata.get("page_label", "unknown"),
                    "type": "text",
                    "format": result_format
                }
            )
            langchain_docs.append(langchain_doc)
        
        return langchain_docs
        
    except Exception as e:
        print(f"LlamaParse 파싱 중 오류 발생: {str(e)}")
        return []

## 3. 텍스트 분할 및 구조화

In [4]:
def clean_metadata(metadata: dict) -> dict:
    """메타데이터에서 Chroma가 지원하지 않는 타입 제거"""
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        elif isinstance(value, list) and all(isinstance(v, (str, int, float)) for v in value):
            # 리스트를 문자열로 변환
            cleaned[key] = ", ".join(str(v) for v in value)
        elif isinstance(value, dict):
            # 딕셔너리를 문자열로 변환
            cleaned[key] = str(value)
        else:
            # 그 외의 타입은 문자열로 변환
            cleaned[key] = str(value)
    return cleaned

def process_llamaparse_documents(documents: List[Document]) -> tuple:
    """LlamaParse로 추출한 문서를 텍스트, 표, 이미지 설명으로 분류"""
    texts = []
    tables = []
    images = []  # 이미지 설명을 담을 리스트
    
    # 텍스트 분할기 초기화
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=4000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    for doc in documents:
        content = doc.page_content
        metadata = clean_metadata(doc.metadata)
        doc_format = metadata.get("format", "markdown")
        
        print(f"문서 형식: {doc_format}")
        
        # 마크다운 표와 이미지 설명 감지
        lines = content.split('\n')
        current_chunk = []
        is_table = False
        
        for line in lines:
            # 이미지 설명 감지 ([이미지: ...] 형태)
            if line.strip().startswith('[이미지:') and line.strip().endswith(']'):
                # 이전 텍스트 청크 처리
                if current_chunk:
                    chunk_content = '\n'.join(current_chunk)
                    if chunk_content.strip():
                        if is_table:
                            tables.append({
                                "content": chunk_content,
                                "metadata": {**metadata, "type": "table"},
                                "type": "table"
                            })
                        else:
                            sub_chunks = text_splitter.split_text(chunk_content)
                            for i, sub_chunk in enumerate(sub_chunks):
                                texts.append({
                                    "content": sub_chunk,
                                    "metadata": {**metadata, "chunk_index": i, "type": "text"},
                                    "type": "text"
                                })
                    current_chunk = []
                    is_table = False
                
                # 이미지 설명 추가
                image_description = line.strip()[7:-1]  # [이미지: 와 ] 제거
                images.append({
                    "content": image_description,
                    "metadata": {**metadata, "type": "image_description"},
                    "type": "image_description"
                })
                continue
            
            # 마크다운 표 형식 감지 (markdown 형식일 때만)
            if doc_format == "markdown" and '|' in line and line.count('|') >= 2:
                if not is_table:
                    # 이전 텍스트 청크 처리
                    if current_chunk:
                        chunk_content = '\n'.join(current_chunk)
                        if chunk_content.strip():
                            # 텍스트를 더 작은 청크로 분할
                            sub_chunks = text_splitter.split_text(chunk_content)
                            for i, sub_chunk in enumerate(sub_chunks):
                                texts.append({
                                    "content": sub_chunk,
                                    "metadata": {**metadata, "chunk_index": i, "type": "text"},
                                    "type": "text"
                                })
                        current_chunk = []
                    is_table = True
                current_chunk.append(line)
            else:
                if is_table and current_chunk:
                    # 표 청크 처리
                    table_content = '\n'.join(current_chunk)
                    if table_content.strip():
                        tables.append({
                            "content": table_content,
                            "metadata": {**metadata, "type": "table"},
                            "type": "table"
                        })
                    current_chunk = []
                    is_table = False
                current_chunk.append(line)
        
        # 마지막 청크 처리
        if current_chunk:
            chunk_content = '\n'.join(current_chunk)
            if chunk_content.strip():
                if is_table:
                    tables.append({
                        "content": chunk_content,
                        "metadata": {**metadata, "type": "table"},
                        "type": "table"
                    })
                else:
                    # 텍스트를 더 작은 청크로 분할
                    sub_chunks = text_splitter.split_text(chunk_content)
                    for i, sub_chunk in enumerate(sub_chunks):
                        texts.append({
                            "content": sub_chunk,
                            "metadata": {**metadata, "chunk_index": i, "type": "text"},
                            "type": "text"
                        })
    
    return texts, tables, images

def extract_pdf_elements_with_llamaparse(pdf_path: str, result_format="markdown") -> tuple:
    """LlamaParse를 사용하여 PDF에서 모든 요소 추출"""
    # LlamaParse로 PDF 파싱
    documents = extract_pdf_with_llama_parse(pdf_path, result_format)
    
    if not documents:
        print("LlamaParse 파싱에 실패했습니다.")
        return [], [], []
    
    # 텍스트, 표, 이미지 설명 분류
    texts, tables, images = process_llamaparse_documents(documents)
    
    return texts, tables, images

## 4. 텍스트와 표 요약 함수

In [5]:
def summarize_table(table_content: str, llm: ChatOpenAI) -> str:
    """표를 구조화된 텍스트로 요약"""
    prompt = f"""다음 표의 내용을 분석하고 요약해주세요:

{table_content}

다음 사항들을 포함해주세요:
1. 표의 주요 내용과 구조
2. 핵심 데이터와 수치
3. 데이터에서 도출할 수 있는 인사이트
4. 표가 전달하고자 하는 주요 메시지"""
    
    response = llm.invoke(prompt)
    return response.content

def enhance_image_description(image_description: str, llm: ChatOpenAI) -> str:
    """LlamaParse로 추출한 이미지 설명을 더 상세하게 개선"""
    prompt = f"""다음은 PDF에서 추출한 이미지 설명입니다:

{image_description}

이 설명을 더 상세하고 구체적으로 개선해주세요:
1. 이미지의 주요 내용과 구성 요소
2. 차트나 그래프인 경우 데이터 트렌드와 주요 수치
3. 표인 경우 주요 데이터와 의미
4. 이미지에서 추출할 수 있는 핵심 인사이트
5. 문서 전체 맥락에서의 이미지 역할

간결하지만 정보가 풍부한 설명을 제공해주세요."""
    
    response = llm.invoke(prompt)
    return response.content

## 5. Multi-Vector Retriever 생성

In [6]:
def create_multi_vector_retriever(texts: List[Dict], tables: List[Dict], images: List[Dict], llm: ChatOpenAI):
    """Multi-vector retriever 생성"""
    # Initialize components
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma(embedding_function=embeddings)
    docstore = InMemoryStore()
    id_key = "doc_id"
    
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key,
    )
    
    # Process texts (LlamaParse에서 추출한 텍스트는 이미 정리됨)
    text_docs = []
    for text in texts:
        doc_id = str(uuid.uuid4())
        doc = Document(
            page_content=text["content"],
            metadata={**text["metadata"], "type": "text", "doc_id": doc_id}
        )
        retriever.docstore.mset([(doc_id, doc)])
        text_docs.append(doc)
    
    # Process tables with summaries
    table_summaries = []
    if tables:
        print(f"표 요약 생성 중... ({len(tables)}개)")
        for i, table in enumerate(tables):
            print(f"표 {i+1}/{len(tables)} 처리 중...")
            doc_id = str(uuid.uuid4())
            summary = summarize_table(table["content"], llm)
            
            # Store original table
            original_doc = Document(
                page_content=table["content"],
                metadata={**table["metadata"], "type": "table", "doc_id": doc_id}
            )
            retriever.docstore.mset([(doc_id, original_doc)])
            
            # Create summary doc for retrieval
            summary_doc = Document(
                page_content=summary,
                metadata={**table["metadata"], "type": "table_summary", "doc_id": doc_id}
            )
            table_summaries.append(summary_doc)
    
    # Process image descriptions with enhancements
    image_summaries = []
    if images:
        print(f"이미지 설명 개선 중... ({len(images)}개)")
        for i, image in enumerate(images):
            print(f"이미지 설명 {i+1}/{len(images)} 처리 중...")
            doc_id = str(uuid.uuid4())
            enhanced_description = enhance_image_description(image["content"], llm)
            
            # Store original image description
            original_doc = Document(
                page_content=image["content"],
                metadata={**image["metadata"], "type": "image_description", "doc_id": doc_id}
            )
            retriever.docstore.mset([(doc_id, original_doc)])
            
            # Create enhanced description doc for retrieval
            summary_doc = Document(
                page_content=enhanced_description,
                metadata={**image["metadata"], "type": "enhanced_image", "doc_id": doc_id}
            )
            image_summaries.append(summary_doc)
    
    # Add all documents to vectorstore with filtered metadata
    all_docs = text_docs + table_summaries + image_summaries
    if all_docs:
        print("벡터스토어에 문서 추가 중...")
        # Filter complex metadata before adding to vectorstore
        filtered_docs = filter_complex_metadata(all_docs)
        retriever.vectorstore.add_documents(filtered_docs)
        print("문서 추가 완료")
    
    return retriever

## 6. RAG Chain 생성

In [7]:
def create_rag_chain(retriever, llm: ChatOpenAI):
    """RAG chain 생성"""
    def format_docs(docs: List[Document]) -> str:
        """문서를 텍스트 형태로 포맷팅"""
        formatted_docs = []
        for doc in docs:
            doc_type = doc.metadata.get("type", "unknown")
            content = doc.page_content
            
            if doc_type == "table":
                formatted_docs.append(f"[표]\n{content}")
            elif doc_type in ["image_description", "enhanced_image"]:
                formatted_docs.append(f"[이미지 설명]\n{content}")
            else:
                formatted_docs.append(f"[텍스트]\n{content}")
        
        return "\n\n---\n\n".join(formatted_docs)
    
    def create_prompt(data: Dict) -> str:
        """텍스트 기반 프롬프트 생성"""
        context = data["context"]
        question = data["question"]
        
        prompt = f"""다음 컨텍스트를 바탕으로 질문에 답해주세요.

질문: {question}

컨텍스트:
{context}

답변할 때 다음 사항을 고려해주세요:
1. 텍스트, 표, 이미지 설명의 정보를 종합적으로 활용
2. 구체적인 데이터나 수치가 있으면 인용
3. 이미지나 차트의 트렌드나 패턴을 설명
4. 정확하고 명확한 답변 제공

답변:"""
        
        return prompt
    
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | RunnableLambda(create_prompt)
        | llm
        | StrOutputParser()
    )
    
    return chain

In [10]:
# PDF 경로 설정
pdf_path = "data/[삼성전자]반기보고서_12.pdf"  # 또는 원하는 PDF 파일 경로

# 파일 존재 확인
import os
if not os.path.exists(pdf_path):
    print(f"경고: {pdf_path} 파일을 찾을 수 없습니다.")

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o", max_tokens=2048, temperature=0)

# 결과 형식 선택 (markdown 또는 text)
result_format = "markdown"  # 변경 가능: "text"
print(f"\n선택된 형식: {result_format}")

# LlamaParse를 사용하여 PDF 요소 추출
print("\nLlamaParse로 PDF에서 요소 추출 중...")
print("=" * 60)
texts, tables, images = extract_pdf_elements_with_llamaparse(pdf_path, result_format)
print("=" * 60)
print(f"추출 완료: {len(texts)}개 텍스트, {len(tables)}개 표, {len(images)}개 이미지 설명")


선택된 형식: markdown

LlamaParse로 PDF에서 요소 추출 중...
LlamaParse로 PDF 파싱 중: data/[삼성전자]반기보고서_12.pdf
결과 형식: markdown
이미지와 텍스트를 함께 처리하고 있습니다...
Started parsing the file under job_id b96304a4-5c15-4d36-b22e-28bef21a5b7b
Started parsing the file under job_id 951c7b80-b822-47d4-88a1-417557d7cccd
..Started parsing the file under job_id db7db16e-b9d3-4585-aa2d-cce55515d142
Started parsing the file under job_id f22e16e3-13fa-434e-97e3-eb3eb7ba0be1
추출 완료: 24개 문서
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
문서 형식: markdown
추출 완료: 49개 텍스트, 17개 표, 17개 이미지 설명


In [ ]:
# texts[0]의 'content'내용 출력

print(texts[0]['content'])
print(tables[0]['content'])

# 경영활동과 관련된 중요한 사항의 발생내용

## 1. 경영활동과 관련된 중요한 사항의 발생
| 연도 | 내용 |
|------|------|
| 2021 | 'Neo QLED TV' 공개 |
|      | 전세계 반도체 업체 최초 전 사업장 '탄소/물/폐기물 저감' 인증 |
|      | 업계 최선단 14나노 EUV DDR5 D램 양산 |
|      | 미국 테일러市에 신규 파운드리 라인 투자 발표 |
|      | BESPOKE 인피니트 라인 공개 |
| 2022 | 세계 최초 3나노 GAA 파운드리 양산 |
|      | 차세대 반도체 R&#x26;D단지 기공식 |
|      | 1Tb 8세대 V낸드 양산 |
|      | 초고화소 기술 집약된 2억 화소 이미지센서 '아이소셀 HP2' 출시 |
|      | 업계 최선단 12나노급 D램 양산 |
| 2023 | 'Galaxy Z 폴드5', 'Galaxy Z 플립5' 공개 |
|      | 현존 최대 용량 32Gb DDR5 D램 개발 |
|      | 생성형 AI '삼성 가우스' 공개 |
|      | 'Galaxy S24 시리즈' 공개 |
|      | 업계 최초 '9세대 V낸드' 양산 |
| 2024 | 'Galaxy Z 폴드6', 'Galaxy Z 플립6' 공개 |
|      | 브랜드가치 사상 첫 1천억 달러 기록, 5년 연속 세계 5위 |
|      | 업계 최초 '24Gb GDDR7 D램' 개발 |
|      | Galaxy S25ㆍS25+ㆍS25 Ultra,S25 Edge 공개 |
| 2025 | 업계 최선단 10나노급 6세대 D램 양산 |
|      | 강력한 CPU/GPU/NPU 통합 성능의 '엑시노스 2500' 공개 |
|      | 세계최초 자발광 최고 주사율 27"QHD 500Hz QD-OLED Display 개발 |


: 

# 자본금 변동사항 및 주식의 총수

## 3. 자본금 변동사항

자본금 변동사항은 기업공시서식 작성기준에 따라 반기보고서에 기재하지 않습니다.
(사업보고서 등에 기재 예정)

## 4. 주식의 총수 등

### 가. 주식의 총수

2025년 반기말 현재 당사가 발행한 기명식 보통주식(이하 "보통주") 및 무의결권 기명식 우선주식(이하 "우선주")의 수는 각각 7,780,466,850주와 1,194,671,350주입니다. 당사는 2025년 반기말 현재까지 보통주 1,860,828,928주와 우선주 378,696,686주를 이사회 결의에 의거하여 이익으로 소각한 바 있습니다. 2025년 반기말 현재 유통주식수는 보통주 5,876,745,450주, 우선주 809,337,676주입니다.

(기준일: 2025년 06월 30일)
(단위: 주, %)
| 구 분                          | 주식의 종류 | 합계               | 비고               | |
|-------------------------------|--------------|--------------------|--------------------|---|
|                               | 보통주       | 우선주             |                    | |
| |. 발행할 주식의 총수        | 20,000,000,000 | 5,000,000,000      | 25,000,000,000      |
| Ⅱ1. 현재까지 발행한 주식의 총수 | 7,780,466,850 | 1,194,671,350      | 8,975,138,200      | |
| 현재까지 감소한 주식의 총수   | 1,860,828,928 | 378,696,686        | 2,239,525,614      | |
| 1. 감자                       | −            |                    |                    | |
| 2. 이익소각                  | 1,860,828,928 | 378,696,686        | 2,239,525,614 자사주소각 | |
| 3. 상환주식의 상환           | −            |                    | −                  | |
| 4. 기타                       | −            |                    | −                  | |
| IV. 발행주식의 총수(II-II)   | 5,919,637,922 | 815,974,664        | 6,735,612,586      | |
| V. 자기주식수                 | 42,892,472   | 6,636,988          | 49,529,460         | |
| VI. 유통주식수(IV-V)         | 5,876,745,450 | 809,337,676        | 6,686,083,126      | |
| VII. 자기주식 보유비율       | 0.7          | 0.8                | 0.7                | |

In [9]:
# Multi-vector retriever 생성
print("\n" + "="*50)
print("Multi-vector retriever 생성 중...")
retriever = create_multi_vector_retriever(texts, tables, images, llm)
print("="*50)
print("Retriever 생성 완료")


Multi-vector retriever 생성 중...


/var/folders/md/zm46y3p12rxfjbmvb0nttrbw0000gn/T/ipykernel_67212/952659787.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=embeddings)


표 요약 생성 중... (1개)
표 1/1 처리 중...
벡터스토어에 문서 추가 중...
문서 추가 완료
Retriever 생성 완료


In [ ]:
# 추출된 내용 미리보기
if texts:
    print("📄 첫 번째 텍스트:")
    print(f"{texts[0]['content']}\n")

if tables:
    print("📊 첫 번째 표:")
    print(f"{tables[0]['content']}\n")

if images:
    print("🖼️ 첫 번째 이미지 설명:")
    print(f"{images[0]['content']}\n")
else:
    print("ℹ️ 추출된 이미지 설명이 없습니다.")

## 7. 전체 시스템 실행

## 8. 질문 및 답변 테스트

In [33]:
# RAG chain 생성
print("\nRAG chain 생성 중...")
rag_chain = create_rag_chain(retriever, llm)
print("RAG chain 생성 완료")


RAG chain 생성 중...


NameError: name 'retriever' is not defined

# 질문 예시
questions = [
    "이 문서의 주요 내용은 무엇인가요?",
    "표에서 가장 중요한 데이터는 무엇인가요?",
    "자사주 보유비율은 얼마인가요?",
    "발행주식의 총수와 유통주식수의 차이점은 무엇인가요?",
    "이익소각한 주식 수는 얼마인가요?"
]

print("\n" + "=" * 80)
print("질문 답변 테스트")
print("=" * 80)

for question in questions:
    print(f"\n질문: {question}")
    print("답변 생성 중...")
    answer = rag_chain.invoke(question)
    print(f"답변: {answer}")
    print("-" * 80)

In [ ]:
# 검색 테스트
test_question = "유통주식수는 얼마인가요?"
retrieved_docs = retriever.get_relevant_documents(test_question)

print(f"질문: {test_question}")
print(f"\n검색된 문서 수: {len(retrieved_docs)}\n")

for i, doc in enumerate(retrieved_docs[:3]):  # 상위 3개만 표시
    print(f"문서 {i+1}:")
    print(f"타입: {doc.metadata.get('type', 'unknown')}")
    print(f"내용 미리보기: {doc.page_content[:200]}...")
    print("-" * 50)

## 9. 대화형 질문 답변

In [ ]:
# 대화형 질문 답변 (원하는 질문을 입력하세요)
while True:
    user_question = input("\n질문을 입력하세요 (종료하려면 'quit' 입력): ")
    if user_question.lower() == 'quit':
        break
    
    print("\n답변 생성 중...")
    answer = rag_chain.invoke(user_question)
    print(f"\n답변: {answer}")

## 10. 검색된 문서 확인 (디버깅용)

In [ ]:
# 상세한 검색 결과 분석
def analyze_retrieval_results(question: str, top_k: int = 5):
    """검색 결과를 상세히 분석하는 함수"""
    print(f"📋 검색 분석: '{question}'")
    print("=" * 60)
    
    retrieved_docs = retriever.get_relevant_documents(question)
    print(f"총 검색된 문서 수: {len(retrieved_docs)}")
    
    # 문서 타입별 분류
    doc_types = {}
    for doc in retrieved_docs:
        doc_type = doc.metadata.get('type', 'unknown')
        if doc_type not in doc_types:
            doc_types[doc_type] = 0
        doc_types[doc_type] += 1
    
    print(f"\n📊 문서 타입별 분포:")
    for doc_type, count in doc_types.items():
        print(f"  - {doc_type}: {count}개")
    
    print(f"\n📄 상위 {min(top_k, len(retrieved_docs))}개 문서 상세:")
    for i, doc in enumerate(retrieved_docs[:top_k]):
        print(f"\n[문서 {i+1}]")
        print(f"타입: {doc.metadata.get('type', 'unknown')}")
        print(f"소스: {doc.metadata.get('source', 'unknown')}")
        print(f"내용 (처음 300자): {doc.page_content[:300]}...")
        if i < len(retrieved_docs) - 1:
            print("-" * 40)

# 분석 실행
analyze_retrieval_results("유통주식수와 자사주 보유비율은?", top_k=3)

In [ ]:
# 최종 시스템 요약
print("\n" + "🎯" * 20)
print("Multi-modal RAG 시스템 구축 완료!")
print("🎯" * 20)

print(f"""
✅ PDF 파싱: LlamaParse 사용
✅ 추출된 요소: {len(texts)}개 텍스트, {len(tables)}개 표, {len(images)}개 이미지
✅ RAG 시스템: Multi-vector retriever + GPT-4o
✅ 상태: 질문 답변 준비 완료

💡 사용 방법:
1. 위의 질문 예시를 실행
2. 대화형 질문 답변 셀 실행
3. 원하는 질문 입력하여 테스트
""")